In [1]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import datetime


# Load my functions
from utils import minmax_normalise_tensor, midpoint_to_box, upscale_box_tensor, box_tensor_2_mid_points
from metrics import mse, rmse
from covariance import covariance_function, aggregate_base_covariance_matrix, predictive_mean

/Users/kimbente/opt/anaconda3/envs/py3.9/lib/python3.9/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: dlopen(/Users/kimbente/opt/anaconda3/envs/py3.9/lib/python3.9/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN2at4_ops19empty_memory_format4callEN3c108ArrayRefIxEENS2_8optionalINS2_10ScalarTypeEEENS5_INS2_6LayoutEEENS5_INS2_6DeviceEEENS5_IbEENS5_INS2_12MemoryFormatEEE
  Referenced from: <67CD63CE-57E0-341F-B3B8-78729B03D2B3> /Users/kimbente/opt/anaconda3/envs/py3.9/lib/python3.9/site-packages/torchvision/image.so
  Expected in:     <18497461-1393-3DF8-BED0-DC986FDB1051> /Users/kimbente/opt/anaconda3/envs/py3.9/lib/python3.9/site-packages/torch/lib/libtorch_cpu.dylib
  warn(f"Failed to load image Python extension: {e}")


# Load scene

Scene target tensor: [N, C, H, W]  

with channels: 
- `[:, 0, :, :]` bed
- `[:, 1, :, :]` surface
- `[:, 2, :, :]` thickness
- `[:, 3, :, :]` mask
- `[:, 4, :, :]` firn
- `[:, 5, :, :]` errorbed

and polar stereographic coordinates
- `[:, 6, :, :]` y
- `[:, 7, :, :]` x

# Language/terminology
- low resolution input tensor
- target tensor
  - inferred
  - ground truth
- auxiliary tensor

In [2]:
scene_bed_tensor = torch.load('./torch_data/DOMEC_bed_scenes.pt')
# scene_bed_tensor = torch.load('./torch_data/TRANSANT_bed_scenes.pt')

## Check correlations of variables

Correlations are not clear: However, how are covariances related? 

In [3]:
# Create format needed for torch.corrcoef() where every variable is presented by a row vector
# Flatten scenes and make variables columns
flat_vars = torch.cat((scene_bed_tensor[:, 0, :, :].reshape(-1).unsqueeze(0),
                       scene_bed_tensor[:, 1, :, :].reshape(-1).unsqueeze(0),
                       scene_bed_tensor[:, 2, :, :].reshape(-1).unsqueeze(0),
                       scene_bed_tensor[:, 4, :, :].reshape(-1).unsqueeze(0)), dim = 0)

# Affirm shape
print(flat_vars.shape)

torch.corrcoef(flat_vars)

torch.Size([4, 1822500])


tensor([[ 1.0000, -0.2246, -0.9278, -0.4568],
        [-0.2246,  1.0000,  0.5719,  0.8969],
        [-0.9278,  0.5719,  1.0000,  0.7279],
        [-0.4568,  0.8969,  0.7279,  1.0000]])

Observations

- Correlation between variables are very different between mountainous and non-mountainous areas.

Mountainous domain:
- Bed and surface are pos. correlated (.6) in the mountainous area. Since surface >= bed, this makes sense.
- Bed and thickness are mildly neg. corralated (-.1) in the mountainous area.
  - Split into pos. and neg. bed elevation and run again.
- Surface and thickness are .7 in the mountainous areas.

Dome C domain:
- Neg. correlations between bed and surface (- .22), bed and thickness (- .93), bed and firn.
- Surface and thickness are .57 

In [ ]:
scene_bed_tensor[0, 0, :, :].unsqueeze(0)

upscale = torch.nn.AvgPool2d(kernel_size = 2, padding = 0)
upscale(scene_bed_tensor[0, 0, :, :].unsqueeze(0)).shape

In [4]:
# Define upscaling factor
# Or use 420 x 420 images and run 2 - 7
up_factor_list = [2, 3, 4, 5, 6, 7, 8, 9]
# up_factor_list = [5]
n_scenes = 20 # max 900, 200

# For loop first and wrap up into a function once it works
loss_df = pd.DataFrame(index = range(0, n_scenes), columns = ['up_2', 'up_3', 'up_4', 'up_5', 'up_6', 'up_7', 'up_8'])
baseline_loss_df = pd.DataFrame(index = range(0, n_scenes), columns = ['up_2', 'up_3', 'up_4', 'up_5', 'up_6', 'up_7', 'up_8'])

# loss_df = pd.DataFrame(index = range(0, n_scenes), columns = ["up_5"])
# baseline_loss_df = pd.DataFrame(index = range(0, n_scenes), columns = ["up_5"])


# for each scene (or fist n_scenes)
for u_index, u in enumerate(up_factor_list):
    for i in range(0, n_scenes):
        # Normalise target bed topography scene and create explicit first dim, e.g. torch.Size([1, 45, 45])
        target_ground_truth = minmax_normalise_tensor(scene_bed_tensor[i, 0, :, :]).unsqueeze(0)

        # Generate box channels for target scene: e.g. from torch.Size([2, 45, 45]) to torch.Size([4, 45, 45])
        target_box_channels = midpoint_to_box(torch.cat((scene_bed_tensor[i, 6, :, :].unsqueeze(0), 
                                                        scene_bed_tensor[i, 7, :, :].unsqueeze(0)), dim = 0))
        
        # Ground truth box: Concatenate bed_elevation channel and mid_points for upscaling function
        target_ground_truth_box = torch.cat((target_ground_truth, target_box_channels), dim = 0)
        
        # Upscale (Increase scale of each pixel, reduce resolution) to generate low-res. input
        lr_bed = upscale_box_tensor(target_ground_truth_box, upscaling_factor = u)
        
        # Subset last 4 channels to get the box channels of the low-resolution grid
        lr_box_channels = lr_bed[-4:, :, :]

        # Normalisation of high-resolution auxiliary channel to compute the base_covariance
        hr_aux = minmax_normalise_tensor(scene_bed_tensor[i, 1, :, :]).unsqueeze(0)

        ### Base covariance ###
        # In non-aligning grids this is the covariance matrix on the AUXILIARY grid not on the target grid
        # Thus non-aligning cases need an additional step of creating the base covariance on the appropriate grid
        # Covariance function: No need to use true spatial coordinates as we can add these channels afterwards. 
        # Parameters like the lengthscale operate on the normalised space
        base_covariance = covariance_function(hr_aux)

        # Add box channels in orinigal Polar steorgraphic coordinates
        # Flatten from 2D to 1D, cast new dim and repeat
        row_box_channels = target_box_channels.reshape(4, -1).unsqueeze(-1).repeat(1, 1, base_covariance.shape[-1])
        # .unsqueeze(-2) creates explicit dimension we wanna copy across (middle dimension)
        column_box_channels = target_box_channels.reshape(4, -1).unsqueeze(-2).repeat(1, base_covariance.shape[-1], 1)

        # generate base covariance box with Channel 0: covar, Channel 1:4: row, Channel 5:8: column box channels. 
        base_covar_box = torch.cat((torch.tensor(base_covariance).unsqueeze(0), row_box_channels, column_box_channels), dim = 0)

        k_ah_al_tensor, k_al_al_tensor = aggregate_base_covariance_matrix(base_covar_box, lr_box_channels)

        ### MEAN RECONSTRUCTION ###
        hr_inferred = predictive_mean(lr_bed[0, :, :].unsqueeze(0), k_ah_al_tensor, k_al_al_tensor, noise = 0.05, mu = 0.5)

        ### LOSS ###
        # inplace mutation of row i and column u (upscale_factor) of df
        loss_df.iloc[i, u_index] = rmse(hr_inferred, target_ground_truth.squeeze()).numpy().item()

        ### BASELINE ###
        # For torch grid resample function: Normalised grid as image input [N, C, H, W] where N = 1 and C = 1. 
        # H_in and W_in are implicit: corners of midpoints are assumed to me -1, -1 (top left) and 1, 1 (bottom right).
        # Always first dim
        lr_input_grid = lr_bed[0, :, :].unsqueeze(0).unsqueeze(0)

        # Generate query grid in relation to input lr grid
        # input uses midpoint notation
        lr_midpoints = box_tensor_2_mid_points(lr_box_channels)

        # retrieve min and max of midpoints to normalise target grid points in relation to lr
        lr_midpoint_y_min = torch.min(lr_midpoints[0, :, :])
        lr_midpoint_y_max = torch.max(lr_midpoints[0, :, :])
        lr_midpoint_x_min = torch.min(lr_midpoints[1, :, :])
        lr_midpoint_x_max = torch.max(lr_midpoints[1, :, :])

        # target_grid: min max normalise in relation. Then double range and subtract 0 to cover [-1, 1]
        y_target_grid = (((scene_bed_tensor[i, 6, :, :] - lr_midpoint_y_min)/(lr_midpoint_y_max - lr_midpoint_y_min) * 2) - 1).unsqueeze(0).unsqueeze(-1)
        # y_target_grid[0, :, 0, 0]

        # Flip y query location because of torch images the origin is in the topleft corner, not in the lower left
        y_target_grid = torch.flip(y_target_grid, dims = (1,))
        
        x_target_grid = (((scene_bed_tensor[i, 7, :, :] - lr_midpoint_x_min)/(lr_midpoint_x_max - lr_midpoint_x_min) * 2) - 1).unsqueeze(0).unsqueeze(-1)
        # x_target_grid[0, 0, :, 0]

        # Flip y-dimension because of torch images the origin is in the topleft corner, not in the lower left
        # SOMEHOW X, Y ?!!
        target_grid = torch.concat((x_target_grid, y_target_grid), dim = -1)
        
        hr_bilinear = torch.nn.functional.grid_sample(lr_input_grid, target_grid, mode = 'bilinear', padding_mode = 'border', align_corners = True)
        hr_bilinear = hr_bilinear.squeeze()

        ### BASELINE LOSS ###
        baseline_loss_df.iloc[i, u_index] = rmse(hr_bilinear, target_ground_truth.squeeze()).numpy().item()

ACHTUNG: Upscaling is not closed/has remainder: Mean aggregation is over fields of different sizes. Consider using a different magnification factor.


/var/folders/mq/9lm4wx5x1dg5n49x3vw8bsqc0000gn/T/ipykernel_70907/3521405897.py:51: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  base_covar_box = torch.cat((torch.tensor(base_covariance).unsqueeze(0), row_box_channels, column_box_channels), dim = 0)


ACHTUNG: Upscaling is not closed/has remainder: Mean aggregation is over fields of different sizes. Consider using a different magnification factor.


KeyboardInterrupt: 

In [ ]:
baseline_loss_df.mean()

In [ ]:
loss_df.mean()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x = list(range(2, 8 +1)), y =  baseline_loss_df.mean(), mode = 'lines+markers', name = "Bilinear baseline"))
fig.add_trace(go.Scatter(x = list(range(2, 8 +1)), y = loss_df.mean(), mode = 'lines+markers', name = "Bayesian Fusion using auxiliary surface data"))
fig.update_layout(title = "Reconstruction loss [RMSE] compared to baseline - 200 scenes near Dome C")
fig.update_xaxes(title_text = 'Upscaling factor')
fig.update_yaxes(title_text = 'RMSE')
fig.show()

In [ ]:
def bayesian_fusion_one_scene(i, up_factor, noise = 0.05, mu = 0.5, lambda_s = 0.3, lambda_p = 0.4, sigma_f = 1.0):
    
    target_ground_truth = minmax_normalise_tensor(scene_bed_tensor[i, 0, :, :]).unsqueeze(0)

    # Generate box channels for target scene: e.g. from torch.Size([2, 45, 45]) to torch.Size([4, 45, 45])
    target_box_channels = midpoint_to_box(torch.cat((scene_bed_tensor[i, 6, :, :].unsqueeze(0), 
                                                     scene_bed_tensor[i, 7, :, :].unsqueeze(0)), dim = 0))
    
    # Ground truth box: Concatenate bed_elevation channel and mid_points for upscaling function
    target_ground_truth_box = torch.cat((target_ground_truth, target_box_channels), dim = 0)
    
    # Upscale (Increase scale of each pixel, reduce resolution) to generate low-res. input
    lr_bed = upscale_box_tensor(target_ground_truth_box, upscaling_factor = up_factor)
    
    # Subset last 4 channels to get the box channels of the low-resolution grid
    lr_box_channels = lr_bed[-4:, :, :]

    # Normalisation of high-resolution auxiliary channel to compute the base_covariance
    hr_aux = minmax_normalise_tensor(scene_bed_tensor[i, 1, :, :]).unsqueeze(0)

    ### Base covariance ###
    # In non-aligning grids this is the covariance matrix on the AUXILIARY grid not on the target grid
    # Thus non-aligning cases need an additional step of creating the base covariance on the appropriate grid
    # Covariance function: No need to use true spatial coordinates as we can add these channels afterwards. 
    # Parameters like the lengthscale operate on the normalised space
    base_covariance = covariance_function(hr_aux, lambda_s, lambda_p, sigma_f)

    # Add box channels in orinigal Polar steorgraphic coordinates
    # Flatten from 2D to 1D, cast new dim and repeat
    row_box_channels = target_box_channels.reshape(4, -1).unsqueeze(-1).repeat(1, 1, base_covariance.shape[-1])
    # .unsqueeze(-2) creates explicit dimension we wanna copy across (middle dimension)
    column_box_channels = target_box_channels.reshape(4, -1).unsqueeze(-2).repeat(1, base_covariance.shape[-1], 1)

    # generate base covariance box with Channel 0: covar, Channel 1:4: row, Channel 5:8: column box channels. 
    base_covar_box = torch.cat((torch.tensor(base_covariance).unsqueeze(0), row_box_channels, column_box_channels), dim = 0)

    k_ah_al_tensor, k_al_al_tensor = aggregate_base_covariance_matrix(base_covar_box, lr_box_channels)

    ### MEAN RECONSTRUCTION ###
    hr_inferred = predictive_mean(lr_bed[0, :, :].unsqueeze(0), k_ah_al_tensor, k_al_al_tensor, noise = 0.05, mu = 0.5)

    ### LOSS ###
    mse_scene = mse(hr_inferred, target_ground_truth.squeeze())

    ### BASELINE ###
    # For torch grid resample function: Normalised grid as image input [N, C, H, W] where N = 1 and C = 1. 
    # H_in and W_in are implicit: corners of midpoints are assumed to me -1, -1 (top left) and 1, 1 (bottom right).
    # Always first dim
    lr_input_grid = lr_bed[0, :, :].unsqueeze(0).unsqueeze(0)

    # Generate query grid in relation to input lr grid
    # input uses midpoint notation
    lr_midpoints = box_tensor_2_mid_points(lr_box_channels)

    # retrieve min and max of midpoints to normalise target grid points in relation to lr
    lr_midpoint_y_min = torch.min(lr_midpoints[0, :, :])
    lr_midpoint_y_max = torch.max(lr_midpoints[0, :, :])
    lr_midpoint_x_min = torch.min(lr_midpoints[1, :, :])
    lr_midpoint_x_max = torch.max(lr_midpoints[1, :, :])

    # target_grid: min max normalise in relation. Then double range and subtract 0 to cover [-1, 1]
    y_target_grid = (((scene_bed_tensor[i, 6, :, :] - lr_midpoint_y_min)/(lr_midpoint_y_max - lr_midpoint_y_min) * 2) - 1).unsqueeze(0).unsqueeze(-1)
    
    # Flip y query location because of torch images the origin is in the topleft corner, not in the lower left
    y_target_grid = torch.flip(y_target_grid, dims = (1,))
    # y_target_grid[0, 0, :, 0]
    
    x_target_grid = (((scene_bed_tensor[i, 7, :, :] - lr_midpoint_x_min)/(lr_midpoint_x_max - lr_midpoint_x_min) * 2) - 1).unsqueeze(0).unsqueeze(-1)
    # x_target_grid[0, 0, :, 0]

    # It works when we change the order here?!
    target_grid = torch.concat((x_target_grid, y_target_grid), dim = -1)
    
    # padding_mode = 'zeros': has zero values beyond [-1, 1]
    hr_bilinear = torch.nn.functional.grid_sample(lr_input_grid, target_grid, mode = 'bilinear', padding_mode = 'border', align_corners = True) # align_corners = True
    hr_bilinear = hr_bilinear.squeeze()

    ### BASELINE LOSS ###
    mse_baseline = mse(hr_bilinear, target_ground_truth.squeeze()).numpy().item()

    return hr_inferred, hr_bilinear, target_ground_truth.squeeze(), lr_bed[0, :, :], hr_aux  # mse_scene, mse_baseline

In [ ]:
"""
results = torch.cat((torch.tensor(loss_list).unsqueeze(0),
           torch.tensor(baseline_loss_list).unsqueeze(0)))

filename = datetime.datetime.now().strftime("./experiments/%Y-%m-%d_DomeC_5up_mse_noHPtuning.pt")
torch.save(results, filename)
"""

In [ ]:
print(f"Bayesian Fusion: Mean loss over 100 scenes: {np.mean(loss_list)}")
print(f"Bilinear interpolation: Mean loss over 100 scenes: {np.mean(baseline_loss_list)}")

In [ ]:
# Index 36: both methods struggle
# Index 6: both methods are also not great: See if lengthscale helps (6 and 36 close to each other i think)
# Index 76: bilinear interpolation not great, Bayeisan fusion good due to spatial kernel, not auxiliary input
hr_inferred, hr_bilinear, hr_ground_truth, lr_input, hr_aux = bayesian_fusion_one_scene(i = 1, up_factor = 5, noise = 0.05, mu = 0.5, lambda_s = 0.3, lambda_p = 0.03, sigma_f = 1.)
# lambda_s = 0.3, lambda_p = 0.4, sigma_f = 1.0
# lambda_s: larger values make it less sparse
# sigma_f if we normalise?
# Noise doesn't change the mean

fig = make_subplots(rows = 2, cols = 3,
                    subplot_titles = ("HR bed topo inferred", "HR bed topo bilinear", "ground truth", 
                                      "LR bed topo input", "(HR bed topo inferred - ground truth)", "Auxiliary HR surface elevation"))

fig.add_trace(go.Heatmap(z = hr_inferred, 
                           colorscale = 'haline', zmin = 0, zmax = 1),
                           row = 1, col = 1)

fig.add_trace(go.Heatmap(z = hr_bilinear, 
                           colorscale = 'haline', zmin = 0, zmax = 1),
                           row = 1, col = 2)

fig.add_trace(go.Heatmap(z = hr_ground_truth, 
                           colorscale = 'haline', zmin = 0, zmax = 1),
                           row = 1, col = 3)

fig.add_trace(go.Heatmap(z = lr_input, 
                           colorscale = 'haline', zmin = 0, zmax = 1),
                           row = 2, col = 1)

fig.add_trace(go.Heatmap(z = hr_inferred - hr_ground_truth, 
                           colorscale = 'RdPu', zmin = 0, zmax = 1),
                           row = 2, col = 2)

fig.add_trace(go.Heatmap(z = hr_aux.squeeze(), 
                           colorscale = 'haline', zmin = 0, zmax = 1),
                           row = 2, col = 3)

fig.update_layout(autosize = False, height = 800, width = 1200, font_family = "Helvetica, sans-serif")
fig.update_layout(plot_bgcolor = 'rgba(0, 0, 0, 0)')
# matrix style
fig.update_yaxes(autorange = "reversed")
fig.update_traces(showscale = False)
fig.show()

print(f"RMSE of Bayesian fusion: {rmse(hr_inferred, hr_ground_truth)}")
print(f"RMSE of Bilinear interpolation: {rmse(hr_bilinear, hr_ground_truth)}")

- Image 6 is very interesting: 
  - lengthscale would help

- Shop MSE calculation of both images.
- Is it better if covariance spreads from -1 to 1?

In [ ]:
# 332
hr_inferred, hr_bilinear, hr_ground_truth, lr_input, hr_aux = bayesian_fusion_one_scene(i = 864, up_factor = 5, noise = 0.05, mu = 0.5, lambda_s = 0.3, lambda_p = 0.03, sigma_f = 1.)
# lambda_s = 0.3, lambda_p = 0.4, sigma_f = 1.0
# lambda_s: larger values make it less sparse
# sigma_f if we normalise?
# Noise doesn't change the mean

fig = make_subplots(rows = 2, cols = 3,
                    subplot_titles = ("HR bed topo inferred", "HR bed topo bilinear", "ground truth", 
                                      "LR bed topo input", "(HR bed topo inferred - ground truth)", "Auxiliary HR surface elevation"))

fig.add_trace(go.Heatmap(z = hr_inferred, 
                           colorscale = 'haline', zmin = 0, zmax = 1),
                           row = 1, col = 1)

fig.add_trace(go.Heatmap(z = hr_bilinear, 
                           colorscale = 'haline', zmin = 0, zmax = 1),
                           row = 1, col = 2)

fig.add_trace(go.Heatmap(z = hr_ground_truth, 
                           colorscale = 'haline', zmin = 0, zmax = 1),
                           row = 1, col = 3)

fig.add_trace(go.Heatmap(z = lr_input, 
                           colorscale = 'haline', zmin = 0, zmax = 1),
                           row = 2, col = 1)

fig.add_trace(go.Heatmap(z = hr_inferred - hr_ground_truth, 
                           colorscale = 'RdPu', zmin = 0, zmax = 1),
                           row = 2, col = 2)

fig.add_trace(go.Heatmap(z = hr_aux.squeeze(), 
                           colorscale = 'haline', zmin = 0, zmax = 1),
                           row = 2, col = 3)

fig.update_layout(autosize = False, height = 800, width = 1200, font_family = "Helvetica, sans-serif")
fig.update_layout(plot_bgcolor = 'rgba(0, 0, 0, 0)')
# matrix style
fig.update_yaxes(autorange = "reversed")
fig.update_traces(showscale = False)
fig.show()

print(f"RMSE of Bayesian fusion: {rmse(hr_inferred, hr_ground_truth)}")
print(f"RMSE of Bilinear interpolation: {rmse(hr_bilinear, hr_ground_truth)}")

# Visualise 

x: index
y: loss


## Baseline

Documentation: https://pytorch.org/docs/stable/generated/torch.nn.functional.grid_sample.html

- Read into corner sligning again
- y-flipping needed
- slightly outside [-1, 1] due to higher res.
  - unfair comparison? Mybe take larger scene input?
- Alternative: torch.nn.functional.interpolate(input, size = None, scale_factor = None, mode = 'nearest', align_corners = None, recompute_scale_factor = None, antialias = False)
- Always normalised input reference grid


In [ ]:
import plotly.graph_objects as go

In [ ]:
fig = go.Figure(data = go.Scatter(x = list(range(0, 900)), y = loss_list, mode = 'markers', name = "our method"))
fig.add_trace(go.Scatter(x = list(range(0, 900)), y = baseline_loss_list, mode = 'markers', name = "baseline"))
fig.show()

In [ ]:
fig = go.Figure(data = go.Scatter(x = list(range(0, 100)), y = loss_list, mode = 'markers', name = "our method"))
fig.show()

- Visualise
- Data Frame with different metrics
- Discard edges from loss for fair comparison

In [ ]:
"""
Old code: Normalising into the wrong direction: need to normalise w.r.t. input
# Y values
torch.max(scene_bed_tensor[i, 6, :, :])
torch.min(scene_bed_tensor[i, 6, :, :])

# y grid between -1, 1 
# Unsqueeze first and last to concat
y_target_grid = (((scene_bed_tensor[i, 6, :, :] - torch.min(scene_bed_tensor[i, 6, :, :]))/(torch.max(scene_bed_tensor[i, 6, :, :]) - torch.min(scene_bed_tensor[i, 6, :, :])) * 2) - 1).unsqueeze(0).unsqueeze(-1)
x_target_grid = (((scene_bed_tensor[i, 7, :, :] - torch.min(scene_bed_tensor[i, 7, :, :]))/(torch.max(scene_bed_tensor[i, 7, :, :]) - torch.min(scene_bed_tensor[i, 7, :, :])) * 2) - 1).unsqueeze(0).unsqueeze(-1)

# Concat along last dim [n, H_out, W_out, 2]
target_grid = torch.concat((y_target_grid, x_target_grid), dim = -1)

lr_midpoints = box_tensor_2_mid_points(lr_box_channels)

# Normalise with min-max from hr
# lr_y_norm = (((lr_midpoints[0, :, :] - torch.min(scene_bed_tensor[i, 6, :, :]))/(torch.max(scene_bed_tensor[i, 6, :, :]) - torch.min(scene_bed_tensor[i, 6, :, :])) * 2) - 1).unsqueeze(0)
# Select only one colmn (x) to represent all y variability
lr_y_norm = (((lr_midpoints[0, :, 0] - torch.min(scene_bed_tensor[i, 6, :, :]))/(torch.max(scene_bed_tensor[i, 6, :, :]) - torch.min(scene_bed_tensor[i, 6, :, :])) * 2) - 1).unsqueeze(0)
# lr_x_norm = (((lr_midpoints[1, :, :] - torch.min(scene_bed_tensor[i, 7, :, :]))/(torch.max(scene_bed_tensor[i, 7, :, :]) - torch.min(scene_bed_tensor[i, 7, :, :])) * 2) - 1).unsqueeze(0)
# Select only one row (y) to represent all x variability
lr_x_norm = (((lr_midpoints[1, 0, :] - torch.min(scene_bed_tensor[i, 7, :, :]))/(torch.max(scene_bed_tensor[i, 7, :, :]) - torch.min(scene_bed_tensor[i, 7, :, :])) * 2) - 1).unsqueeze(0)

# Dim torch.Size([1, 9, 9])
lr_midpoints_norm = torch.concat((lr_y_norm, lr_x_norm), dim = 0)
# Concat with
"""

## Questions:

- Normalisation outside or inside
- Mean function

Limitation:
- Transferability of covariance likely depends on scale: new proof of concept for 